In [ ]:
import torch, torchvision, time
from torchvision import transforms
import torch.nn as nn




In [ ]:

def printLine(line):
    print(line)
    outFile = open("/content/drive/MyDrive/ColabNotebooks/logging.csv", "a")
    outFile.write(line + "\n")

#this is used to get the data loaders for each of the sets
def getLoaders(dataset, batchSize):
    if dataset == "mnist":
        mnistTransform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))])
        fullDataSet = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=mnistTransform)
        train, val = torch.utils.data.random_split(fullDataSet, [.83, .17])
        test = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=mnistTransform)
    if dataset == "mnistTest":
        mnistTransform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.1307,), (0.3081,))])
        fullDataSet = torchvision.datasets.MNIST(root="./data", train=True, download=True, transform=mnistTransform)
        train, val = torch.utils.data.random_split(fullDataSet, [.83, .17])
        test = torchvision.datasets.MNIST(root="./data", train=False, download=True, transform=mnistTransform)
        train = fullDataSet
    if dataset == "cifar10Test":
        cifarTransform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010))])
        fullDataSet = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=cifarTransform)
        train, val = torch.utils.data.random_split(fullDataSet, [.9, .1])
        test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=cifarTransform)
        train = fullDataSet
    else:
        cifarTransform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010))])
        fullDataSet = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=cifarTransform)
        train, val = torch.utils.data.random_split(fullDataSet, [.9, .1])
        test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=cifarTransform)

    mk = lambda ds: torch.utils.data.DataLoader(ds, batch_size=batchSize, pin_memory=True)
    return mk(train), mk(val), mk(test)

In [ ]:
#these functions are used to train the model
def trainModel(model, trainSet, valSet, learningRate, epochs, device, optimizer, stoppingFactor):
    print("Training Begining")
    model.to(device)
    if optimizer == "Adam":
        opt = torch.optim.Adam(model.parameters(), lr=learningRate)
    else:
        opt = torch.optim.SGD(model.parameters(), lr=learningRate)
    criteria = torch.nn.CrossEntropyLoss()
    #best index 0 = accuracy, 1 = model state, 2 = epoch
    best = [0, None, 0]
    noImprove = 0
    for epoc in range(1, epochs+1):
        for x,y in trainSet:
            x,y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = criteria(logits, y)
            loss.backward()
            opt.step()

        valAccuracy = evaluate(model, valSet, device)
        if valAccuracy > best[0]:
            best = [valAccuracy, model.state_dict(), epoc]
            noImprove = 0
        else:
            noImprove += 1
        if noImprove == stoppingFactor:
            break
        print(epoc, best[0], valAccuracy)

    if best[1] != None:
        model.load_state_dict(best[1])
    return model, best


def evaluate(model, valSet, device):
    model.eval()
    correct = 0
    n = 0
    for x,y in valSet:
        x,y = x.to(device), y.to(device)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += y.size(0)
    return correct / n

In [ ]:
#defining the cnn class
class BaselineCNN(nn.Module):
    def __init__(self, numberOfChannels, inputHeightWidth, ):
        super().__init__()
        H, W = inputHeightWidth
        self.features = nn.Sequential(
            nn.Conv2d(numberOfChannels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 32*2, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, numberOfChannels, H, W)
            feat = self.features(dummy)
            fc_in = feat.numel()

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_in, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class EnhancedCNN(nn.Module):
    def __init__(self, numberOfChannels, inputHeightWidth, dropRate):
        super().__init__()
        H, W = inputHeightWidth
        self.features = nn.Sequential(
            nn.Conv2d(numberOfChannels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 32*2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32*2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, numberOfChannels, H, W)
            fc_in = self.features(dummy).numel()

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropRate) if dropRate > 0 else nn.Identity(),
            nn.Linear(fc_in, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


class DeeperCNN(nn.Module):
    def __init__(self, numberOfChannels, inputHeightWidth, dropRate):
        super().__init__()
        H, W = inputHeightWidth
        self.features = nn.Sequential(
            nn.Conv2d(numberOfChannels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 32*2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32*2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32*2, 32*4, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32*4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, numberOfChannels, H, W)
            fc_in = self.features(dummy).numel()

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropRate) if dropRate > 0 else nn.Identity(),
            nn.Linear(fc_in, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


In [ ]:
learningRates = [0.01,0.001,0.0001]
batchSize = [32,64,128]
optomizer = ["SGD", "Adam"]
dropRates = [0,0.2,0.5]

lr = learningRates[1]
batchSz = batchSize[1]
optom = optomizer[1]
drpRate = dropRates[1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("The device being used is: " + str(device))
torch.manual_seed(56)
torch.cuda.manual_seed_all(56)

# train, val, test = getLoaders("cifar10", batchSz)

# model = BaselineCNN(1,[28,28])
# model = EnhancedCNN(1, [28, 28], drpRate)
model = DeeperCNN(1, [28, 28], drpRate)

train, val, test = getLoaders("mnist", batchSz)
# t0 = time.time()
trainedmodel, bestArr = trainModel(model,train, val, lr, 20, device, optom, 10)
# runtime = (time.time()-t0)/60
# print(bestArr[0], bestArr[2], runtime)
# printLine("Deeper" + str(lr) + "," + str(batchSz) + "," + str(optom) + "," + str(drpRate) + "," + str(bestArr[0]) + "," + str(runtime))

finalAccuracy = evaluate(trainedmodel, test, device)
print(finalAccuracy)





# for modelnum in [0,1,2]:
#   print("Testing model: " + str(modelnum))
#   for opt in optomizer:
#     print("Testing optomizer: " + opt)
#     for i in [0,1,2]:

#       if modelnum == 0:
#         text = "Baseline,"
#         model = BaselineCNN(3,[32,32])
#       elif(modelnum == 1):
#         text = "Enhanced,"
#         model = EnhancedCNN(3, [32,32], dropRates[i])
#       else:
#         text = "Deeper,"
#         model = DeeperCNN(3, [32,32], dropRates[i])

#       print("testing Combonation: " + str(i))
#       train, val, test = getLoaders("cifar10", batchSize[i])
#       t0 = time.time()
#       trainedmodel, bestArr = trainModel(model,train, val, learningRates[i], 20, device, opt, 10)
#       runtime = (time.time()-t0)/60
#       print(bestArr[0], bestArr[2], runtime)


#       if modelnum == 0:
#         printLine(text + str(learningRates[i]) + "," + str(batchSize[i]) + "," + str(opt) + "," + str("N/A") + "," + str(bestArr[0]) + "," + str(runtime))
#       else:
#         printLine(text + str(learningRates[i]) + "," + str(batchSize[i]) + "," + str(opt) + "," + str(dropRates[i]) + "," + str(bestArr[0]) + "," + str(runtime))






The device being used is: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 494kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.60MB/s]


Training Begining
1 0.9872549019607844 0.9872549019607844
2 0.9908823529411764 0.9908823529411764
3 0.9908823529411764 0.9903921568627451
4 0.9908823529411764 0.9831372549019608
5 0.991078431372549 0.991078431372549
6 0.9914705882352941 0.9914705882352941
7 0.9914705882352941 0.9898039215686274
8 0.9914705882352941 0.9882352941176471
9 0.9914705882352941 0.9886274509803922
10 0.9914705882352941 0.989313725490196
11 0.9914705882352941 0.9887254901960785
12 0.9919607843137255 0.9919607843137255
13 0.9919607843137255 0.9900980392156863
14 0.9920588235294118 0.9920588235294118
15 0.9920588235294118 0.9907843137254903
16 0.9920588235294118 0.990686274509804
17 0.9920588235294118 0.9905882352941177
18 0.9920588235294118 0.9904901960784314
19 0.9925490196078431 0.9925490196078431
20 0.9925490196078431 0.9902941176470588
0.9895
